# MENG 417 — Quiz #2
## Carefully checked, step-by-step mathematics and Python verification

**Source:** the uploaded one-page file *“1 Should the first Quiz 2 MENG 417.docx”*. This notebook addresses the professor’s **Question 1(a–c), Question 2(a–b), and Question 3(a)**, in that order. It contains numerical verification and a final conceptual cheat sheet.

**Source and accuracy notes:**

- This file is **Quiz #2**, not Homework #3.
- The professor instructs students to solve the quiz independently, show their work, and submit each question on a separate sheet. Treat this notebook as a worked **study/reference resource**, then prepare your own submission according to those instructions.
- **Question 1(b)** does not specify the perceptron learning rate or a unique update convention; we state the convention and also give the general formula.
- **Question 1(c)** prints a bipolar sigmoid with parameter $\lambda=1$, which is $\tanh(v/2)$, **not** $\tanh(v)$.
- **Question 3(a)** provides the FIR/ReLU neuron and initial weights, but does **not** specify the learning rule or step size. We show the parametric answer and a labelled normalized-LMS example; there is no uniquely determined numerical estimate without an update convention.

All display mathematics uses `$$ ... $$` as requested.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

np.set_printoptions(precision=9, suppress=True)

# Question 1 (35 points)

The professor gives the line

$$
3x_1+2x_2+1=0
$$

and a perceptron with surface normal

$$
\mathbf w=\begin{bmatrix}-3\\-2\end{bmatrix},\qquad b=-1.
$$

The net input to the perceptron is

$$
\boxed{v=\mathbf w^T\mathbf x+b=-3x_1-2x_2-1.}
$$

The line $v=0$ is the decision boundary. The activation is the **signum** function, with label $+1$ for strictly positive $v$ and label $-1$ for strictly negative $v$.

**Boundary convention:** $\operatorname{sgn}(0)$ is not specified; it does not affect the open half-plane answers.

## Question 1(a) — Determine the half-plane with label $y=1$ (10 points)

**Step 1.** Write the decision formula:

$$
y=\operatorname{sgn}(-3x_1-2x_2-1).
$$

**Step 2.** The output is $+1$ exactly when its argument is positive:

$$
-3x_1-2x_2-1>0.
$$

**Step 3.** Multiply by $-1$, reversing the inequality:

$$
\boxed{3x_1+2x_2+1<0.}
$$

Equivalently,

$$
\boxed{x_2<-\frac32x_1-\frac12.}
$$

**Geometric meaning:** the $y=1$ region is the half-plane **below** the line $x_2=-\frac32x_1-\frac12$.

**Check one point.** At $\mathbf x=(-1,0)^T$,

$$
v=-3(-1)-2(0)-1=2>0,
$$

so this point is correctly in the $+1$ half-plane.

In [ ]:
# Q1(a): plot the decision boundary and the two half-planes.
xs = np.linspace(-3.0, 2.5, 350)
line = -1.5 * xs - 0.5
fig, ax = plt.subplots(figsize=(8, 5.3))
ax.plot(xs, line, linewidth=2, label=r"Decision boundary: $3x_1+2x_2+1=0$")
ax.fill_between(xs, -5, line, alpha=0.14, label=r"Class $+1$: $3x_1+2x_2+1<0$")
ax.fill_between(xs, line, 5, alpha=0.08, label=r"Class $-1$: $3x_1+2x_2+1>0$")
ax.scatter([-1], [0], s=65, label="Check point (-1,0)")
ax.set(xlabel=r"$x_1$", ylabel=r"$x_2$", xlim=(-3, 2.5), ylim=(-4, 4),
       title="Q1(a) — Line and Class +1 Half-Plane")
ax.grid(alpha=0.25)
ax.legend(loc="upper right", fontsize=8)
plt.show()
print("Net input at (-1,0):", -3*(-1)-2*0-1)

## Question 1(b) — Perceptron update for $\mathbf x=[1,1]^T$ with label $+1$ (10 points)

The incoming training example is

$$
\mathbf x=\begin{bmatrix}1\\1\end{bmatrix},\qquad t=+1.
$$

**Step 1. Calculate its current net input.**

$$
v=(-3)(1)+(-2)(1)-1=\boxed{-6}.
$$

**Step 2. Determine the current prediction.**

$$
\hat y=\operatorname{sgn}(-6)=\boxed{-1}.
$$

**Step 3. Compare with target.** The target is $t=+1$, so this sample is **misclassified**. A perceptron-training update is therefore required.

**Step 4. Choose and state an update convention.** A standard signed-label perceptron rule is

$$
\boxed{\mathbf w_{\text{new}}=\mathbf w_{\text{old}}+\eta t\mathbf x\quad\text{when }t(\mathbf w^T\mathbf x+b)\le0.}
$$

Here $t=+1$ and $\eta>0$ is the learning rate. The problem does not specify $\eta$, so the **general answer** is

$$
\boxed{\mathbf w_{\text{new}}=\begin{bmatrix}-3+\eta\\-2+\eta\end{bmatrix}.}
$$

If the usual demonstration choice $\eta=1$ is used,

$$
\boxed{\mathbf w_{\text{new}}=\begin{bmatrix}-2\\-1\end{bmatrix}.}
$$

**Bias:** if the training rule updates the bias as well, then $b_{\text{new}}=b_{\text{old}}+\eta t=-1+\eta$, or $0$ for $\eta=1$. The question explicitly asks for the weight vector.

**Convention check:** Some lecture notes define an error-based rule $\mathbf w\leftarrow\mathbf w+\eta(t-\hat y)\mathbf x$. Because $t-\hat y=2$, that yields $(-3+2\eta,-2+2\eta)^T$; it is the same signed-label step if its step size is halved. State the chosen rule in your own written solution.

In [ ]:
# Q1(b): numerical audit.
w0 = np.array([-3.0, -2.0])
b0 = -1.0
x = np.array([1.0, 1.0]); target = 1.0
v = float(w0 @ x + b0)
pred = 1 if v > 0 else -1
assert v == -6.0 and pred == -1
eta = 1.0  # explicitly chosen illustrative step size
w1 = w0 + eta * target * x
b1 = b0 + eta * target
print("Original score:", v, "| Prediction:", pred, "| Target:", target)
print("Updated weights with eta=1:", w1)
print("Optional updated bias:", b1)
assert np.array_equal(w1, [-2.0, -1.0])

## Question 1(c) — Replace signum with the professor’s bipolar sigmoid (15 points)

**Important:** The professor defines the bipolar sigmoid with $\lambda=1$ as

$$
\boxed{f(v)=\frac{1-e^{-v}}{1+e^{-v}}.}
$$

It can also be written as

$$
f(v)=\tanh\left(\frac v2\right),
$$

**not** $\tanh(v)$.

To derive a gradient update, we explicitly use the standard single-example squared-error loss:

$$
\boxed{E(\mathbf w)=\frac12\big(t-f(v)\big)^2},\qquad v=\mathbf w^T\mathbf x+b.
$$

**Step 1. Differentiate the activation.** Put $z=e^{-v}$. Then

$$
f(v)=\frac{1-z}{1+z},\qquad\frac{dz}{dv}=-z.
$$

Using the quotient rule,

$$
\frac{df}{dv}
=\frac{2e^{-v}}{(1+e^{-v})^2}
=\boxed{\frac12\big(1-f(v)^2\big)}.
$$

**Step 2. Apply the chain rule.**

$$
\frac{\partial E}{\partial\mathbf w}
=-(t-f(v))f'(v)\mathbf x.
$$

**Step 3. Apply gradient descent with learning rate $\eta$.**

$$
\boxed{\mathbf w_{\text{new}}
=\mathbf w_{\text{old}}
+\eta(t-f(v))\frac12(1-f(v)^2)\mathbf x.}
$$

**Step 4. Substitute the sample from Question 1(b).** We have $v=-6$, $t=+1$, $\mathbf x=(1,1)^T$, and

$$
f(-6)=\frac{1-e^6}{1+e^6}\approx-0.995054754.
$$

Therefore,

$$
f'(-6)=\frac12\big(1-f(-6)^2\big)\approx0.004933019.
$$

The scalar multiplier (before $\eta$) is

$$
(t-f(-6))f'(-6)\approx\boxed{0.009841642}.
$$

The resulting **weight update equation** is

$$
\boxed{\mathbf w_{\text{new}}
=\begin{bmatrix}-3\\-2\end{bmatrix}
+\eta(0.009841642)\begin{bmatrix}1\\1\end{bmatrix}.}
$$

If $\eta=1$ is used for illustration,

$$
\boxed{\mathbf w_{\text{new}}\approx\begin{bmatrix}-2.990158358\\-1.990158358\end{bmatrix}.}
$$

This is **different from the discrete perceptron update** because the continuous activation contributes its derivative. The update above assumes squared-error loss, which is **not explicitly specified** in the quiz. Other losses lead to different valid gradient expressions.

In [ ]:
# Q1(c): numerical derivative and weight-update verification.
vv = -6.0
f = (1.0 - math.exp(-vv)) / (1.0 + math.exp(-vv))
fp = 0.5 * (1.0 - f**2)
delta_factor = (1.0 - f) * fp
w_c = np.array([-3.0, -2.0]) + 1.0 * delta_factor * np.array([1.0, 1.0])
print(f"bipolar sigmoid f(-6) = {f:.12f}")
print(f"derivative f'(-6) = {fp:.12f}")
print(f"delta scalar (eta=1) = {delta_factor:.12f}")
print("Updated weights (eta=1):", w_c)
assert abs(f - math.tanh(-3.0)) < 1e-12
assert np.allclose(w_c, [-2.990158357826519, -1.9901583578265192])

# Question 2 (30 points)

Professor-provided classes:

$$
\mathcal C_1:\quad(-1,1),\ (1,-1),\ (0,0).
$$

$$
\mathcal C_2:\quad(0,-1),\ (0,1).
$$

Inspecting these points reveals:

$$
\boxed{x_1+x_2=0\quad\text{for every Class 1 example},}
$$

while

$$
\boxed{x_1+x_2\in\{-1,+1\}\quad\text{for Class 2}.}
$$

This gives us a simple classifier with **two boundary detectors**.

In [ ]:
class1 = np.array([[-1.,1.],[1.,-1.],[0.,0.]])
class2 = np.array([[0.,-1.],[0.,1.]])
fig, ax = plt.subplots(figsize=(6.6,5.5))
ax.scatter(class1[:,0], class1[:,1], marker='o', s=100, label='Class 1')
ax.scatter(class2[:,0], class2[:,1], marker='x', s=110, label='Class 2')
grid = np.linspace(-1.7, 1.7, 200)
ax.plot(grid, -grid, linestyle=':', label=r'$x_1+x_2=0$')
ax.set(xlim=(-1.7,1.7), ylim=(-1.7,1.7), xlabel=r'$x_1$', ylabel=r'$x_2$', title='Q2 — Original five points')
ax.grid(alpha=.25); ax.legend(); plt.show()

## Question 2(a) — Can a single perceptron solve the dataset? (5 points)

A single perceptron can perfectly classify two classes only when they are **linearly separable**.

**Step 1. Identify an important Class 1 point:**

$$
(0,0)\in\mathcal C_1.
$$

**Step 2. Consider the two Class 2 points:**

$$
(0,-1),\quad (0,1)\in\mathcal C_2.
$$

Their midpoint is

$$
\frac12(0,-1)+\frac12(0,1)=\boxed{(0,0)}.
$$

Thus a Class 1 point lies inside the convex hull of Class 2. A straight line cannot put both Class 2 endpoints strictly on one side while their midpoint belongs strictly to the opposite side.

**Conclusion:**

$$
\boxed{\text{The data are not linearly separable; a single perceptron cannot classify them perfectly.}}
$$

## Question 2(b) — Design a two-layer neural network and draw its block diagram (25 points)

A **two-layer network** here means two layers of trainable neurons: one hidden layer, then one output layer. The input itself is not counted as a trainable layer.

### Step 1 — First hidden neuron: detect the positive side of the diagonal

$$
a_1=x_1+x_2-\frac12,
$$

$$
\boxed{h_1=H(a_1),\quad\mathbf w_1=[1,1]^T,\quad b_1=-\frac12.}
$$

### Step 2 — Second hidden neuron: detect the negative side

$$
a_2=-x_1-x_2-\frac12,
$$

$$
\boxed{h_2=H(a_2),\quad\mathbf w_2=[-1,-1]^T,\quad b_2=-\frac12.}
$$

Use the hard step activation

$$
H(z)=\begin{cases}1,&z\ge0,\\0,&z<0.\end{cases}
$$

### Step 3 — Output neuron: combine the two detectors using an OR rule

$$
\boxed{\hat y=H\left(h_1+h_2-\frac12\right).}
$$

Let output $0$ mean **Class 1** and output $1$ mean **Class 2**.

### Step 4 — Why it works

Every Class 1 point has $x_1+x_2=0$, so $a_1=a_2=-0.5$ and $(h_1,h_2)=(0,0)$. Output is $0$.

Every Class 2 point has $x_1+x_2=+1$ or $-1$; exactly one hidden neuron activates. Output is $1$.

**The architecture has two hidden neurons and one output neuron, and classifies all five examples.**

In [ ]:
# Q2(b): block diagram of the exact two-layer architecture.
fig, ax = plt.subplots(figsize=(11,5.8))
ax.set_xlim(0, 11); ax.set_ylim(0, 6); ax.axis('off')
node_style = dict(boxstyle='round,pad=0.7', edgecolor='black', facecolor='white', linewidth=1.2)
ax.text(1.3, 4.4, r'$x_1$', ha='center', va='center', bbox=node_style, fontsize=14)
ax.text(1.3, 1.6, r'$x_2$', ha='center', va='center', bbox=node_style, fontsize=14)
ax.text(5.0, 4.3, r'$h_1=H(x_1+x_2-0.5)$', ha='center', va='center', bbox=node_style, fontsize=12)
ax.text(5.0, 1.7, r'$h_2=H(-x_1-x_2-0.5)$', ha='center', va='center', bbox=node_style, fontsize=12)
ax.text(9.2, 3.0, r'$y=H(h_1+h_2-0.5)$', ha='center', va='center', bbox=node_style, fontsize=12)
for p,q,label in [
    ((2.1,4.4),(3.45,4.3),'+1'),((2.1,1.6),(3.45,4.0),'+1'),
    ((2.1,4.4),(3.45,2.0),'-1'),((2.1,1.6),(3.45,1.7),'-1'),
    ((6.8,4.3),(7.65,3.35),'+1'),((6.8,1.7),(7.65,2.65),'+1')]:
    ax.annotate('', xy=q, xytext=p, arrowprops=dict(arrowstyle='->', lw=1.5))
    ax.text((p[0]+q[0])/2,(p[1]+q[1])/2+0.15,label,fontsize=9,ha='center')
ax.set_title('Q2(b) — Two trainable layers: 2 hidden step units + 1 output step unit', pad=15)
plt.show()

In [ ]:
# Q2(b): check all 5 professor-provided points.
def H(z):
    return (np.asarray(z) >= 0).astype(int)

def two_layer_predict(X):
    X = np.atleast_2d(np.asarray(X, dtype=float))
    h1 = H(X[:, 0] + X[:, 1] - 0.5)
    h2 = H(-X[:, 0] - X[:, 1] - 0.5)
    output = H(h1 + h2 - 0.5)
    return h1, h2, output

X = np.vstack((class1,class2))
true = np.array([0,0,0,1,1])
h1,h2,yhat = two_layer_predict(X)
print('Point       Class    h1   h2   Output   Correct')
for pt, t, a, b, pred in zip(X,true,h1,h2,yhat):
    print(f'{str(tuple(pt)):13s}{t:5d}{a:6d}{b:5d}{pred:8d}{str(t==pred):>9s}')
assert np.array_equal(yhat,true)
print('\nAll five training points are correctly classified.')

# Question 3(a) (30 points) — ReLU neuron and adaptive prediction

The professor provides

$$
x[0]=32,\qquad x[1]=16,\qquad x[2]=8,\qquad x[3]=?.
$$

The neuron is

$$
\boxed{\hat x[n+1]=\operatorname{ReLU}(w_1x[n]+w_2x[n-1]).}
$$

Initial weights:

$$
\boxed{w_1^{(0)}=\frac12,\quad w_2^{(0)}=\frac12.}
$$

We must start training by predicting $x[2]$ using $x[1]$ and $x[0]$.

**Source limitation:** The quiz does **not** specify the exact weight-update equation or the learning rate. Therefore there is no uniquely determined numerical result for the learned weights or $x[3]$ unless that additional convention is supplied. We derive the generic answer, then give a clearly labelled normalized-LMS illustrative example.

## Step 1 — Predict the known sample $x[2]$

Take $n=1$, so the regressor is

$$
\mathbf u[1]=\begin{bmatrix}x[1]\\x[0]\end{bmatrix}
=\begin{bmatrix}16\\32\end{bmatrix}.
$$

The initial preactivation is

$$
a[1]=\frac12(16)+\frac12(32)=8+16=\boxed{24}.
$$

Because $24>0$, ReLU leaves the value unchanged:

$$
\boxed{\hat x[2]=\operatorname{ReLU}(24)=24.}
$$

The true sample is $x[2]=8$. Thus the error is

$$
\boxed{e[2]=x[2]-\hat x[2]=8-24=-16.}
$$

## Step 2 — Update the weights (learning-rule assumption)

Using **unnormalized LMS** with learning rate $\mu$,

$$
\mathbf w^{(1)}=\mathbf w^{(0)}+\mu e[2]\mathbf u[1],
$$

where the derivative of ReLU is $1$ at the positive preactivation $a[1]=24$.

Therefore,

$$
\mathbf w^{(1)}
=\begin{bmatrix}1/2\\1/2\end{bmatrix}
+\mu(-16)\begin{bmatrix}16\\32\end{bmatrix}
=\boxed{\begin{bmatrix}1/2-256\mu\\1/2-512\mu\end{bmatrix}}.
$$

## Step 3 — Estimate $x[3]$

The input uses the two newest **observed** samples $x[2]=8$ and $x[1]=16$:

$$
\hat x[3]
=\operatorname{ReLU}(w_1^{(1)}\cdot8+w_2^{(1)}\cdot16).
$$

Substitute the updated weights:

$$
\hat x[3]
=\operatorname{ReLU}\left(8(\tfrac12-256\mu)+16(\tfrac12-512\mu)\right).
$$

Simplify:

$$
\boxed{\hat x[3]=\max(0,12-10240\mu).}
$$

**This is the general answer for the explicitly stated unnormalized LMS assumption.** Without a specified $\mu$, the quiz does not determine one numerical prediction.

## Optional fully worked numerical example: normalized LMS

A common adaptive-filter convention is **normalized LMS (NLMS)** with normalization $\|\mathbf u\|^2$ and normalized step size $\alpha=1$:

$$
\boxed{\mathbf w^{(1)}=\mathbf w^{(0)}+
\alpha\frac{e[2]}{\|\mathbf u[1]\|^2}\mathbf u[1].}
$$

First compute the input energy:

$$
\|\mathbf u[1]\|^2=16^2+32^2=256+1024=\boxed{1280}.
$$

Now compute the correction:

$$
\Delta\mathbf w
=\frac{-16}{1280}\begin{bmatrix}16\\32\end{bmatrix}
=\begin{bmatrix}-0.2\\-0.4\end{bmatrix}.
$$

Updated weights:

$$
\mathbf w^{(1)}
=\begin{bmatrix}0.5\\0.5\end{bmatrix}
+\begin{bmatrix}-0.2\\-0.4\end{bmatrix}
=\boxed{\begin{bmatrix}0.3\\0.1\end{bmatrix}}.
$$

Estimate the next sample:

$$
\hat x[3]=\operatorname{ReLU}(0.3\cdot8+0.1\cdot16)
=\operatorname{ReLU}(2.4+1.6)
=\boxed{4}.
$$

The observed sequence halves $32,16,8$, so $4$ is also a natural extrapolation. **However, the numerical result $4$ depends on the stated normalized-LMS update assumption**; the handout alone does not uniquely select it.

In [ ]:
# Q3(a): reproduce both the general LMS formula and the NLMS example.
x = np.array([32.0,16.0,8.0])
w_init = np.array([0.5,0.5])
u = np.array([x[1],x[0]])
a = float(w_init @ u)
x2_pred = max(0., a)
err = x[2] - x2_pred
print('Initial x[2] prediction:',x2_pred)
print('Prediction error e[2]:',err)

# Generic unnormalized LMS result (choose a mu to illustrate).
mu = 0.0001
w_lms = w_init + mu * err * u
x3_lms = max(0., float(w_lms @ np.array([x[2],x[1]])))
print('Generic LMS formula: x3_hat = max(0, 12 - 10240*mu)')
print('Example mu=0.0001:',x3_lms)
assert abs(x3_lms - max(0,12-10240*mu))<1e-12

# Normalized-LMS example, alpha=1.
alpha=1.0
w_nlms = w_init + alpha * (err/float(u@u)) * u
x3_nlms = max(0., float(w_nlms @ np.array([x[2],x[1]])))
print('NLMS alpha=1 weights:', w_nlms)
print('NLMS alpha=1 x3 prediction:', x3_nlms)
assert np.allclose(w_nlms,[0.3,0.1]) and abs(x3_nlms-4)<1e-12

# Final compact answer sheet (for study/reference)

### Question 1(a)

$$
\boxed{y=+1\iff-3x_1-2x_2-1>0\iff3x_1+2x_2+1<0.}
$$

### Question 1(b)

$$
\boxed{v=-6,\quad\hat y=-1,\quad t=+1.}
$$

The sample is misclassified. Under the signed-label perceptron convention:

$$
\boxed{\mathbf w_{\text{new}}=(-3+\eta,-2+\eta)^T.}
$$

For $\eta=1$, $\mathbf w_{\text{new}}=(-2,-1)^T$.

### Question 1(c)

For the professor’s bipolar sigmoid, $f(v)=\tanh(v/2)$, the squared-error gradient update is

$$
\boxed{\Delta\mathbf w=\eta(t-f(v))\tfrac12(1-f(v)^2)\mathbf x.}
$$

At $v=-6$ and $t=1$, $\Delta\mathbf w\approx\eta\cdot0.009841642(1,1)^T$.

### Question 2(a)

A single perceptron is insufficient: $\mathcal C_1$ contains the midpoint $(0,0)$ of the two $\mathcal C_2$ points.

### Question 2(b)

$$
h_1=H(x_1+x_2-0.5),\quad h_2=H(-x_1-x_2-0.5),
$$

$$
\boxed{\hat y=H(h_1+h_2-0.5).}
$$

Output $0\to$ Class 1, output $1\to$ Class 2. All five examples classify correctly.

### Question 3(a)

First prediction $\hat x[2]=24$, error $e[2]=-16$. Under unnormalized LMS,

$$
\boxed{\hat x[3]=\max(0,12-10240\mu).}
$$

Under normalized LMS with $\alpha=1$, final weights $[0.3,0.1]^T$ and $\boxed{\hat x[3]=4}$.

# Theory cheat sheet — five ideas to remember

**1. The decision half-plane**

$$
y=+1\iff\mathbf w^T\mathbf x+b>0.
$$

**2. The signed-label perceptron update**

$$
\mathbf w\leftarrow\mathbf w+\eta t\mathbf x
\quad\text{for a misclassified sample}.
$$

**3. The professor’s bipolar sigmoid has a half-sized tanh argument**

$$
\frac{1-e^{-v}}{1+e^{-v}}=\tanh(v/2),
\qquad f'(v)=\frac12(1-f(v)^2).
$$

**4. Two layers can combine multiple linear boundaries.** Two hidden threshold units detect opposite sides of a band; an output threshold implements OR.

**5. Adaptive prediction must state its learning rule.** ReLU is $\max(0,z)$, but an initial weight vector alone is not enough to specify a unique trained prediction. Always state the learning rate and whether the update is LMS or normalized LMS.

## Before preparing your own Canvas submission

The professor asks for handwritten work (one sheet per question) and explicitly requires independent solutions. Check the learning-rate and loss convention in your own lecture notes, especially for Questions **1(b)**, **1(c)**, and **3(a)**. Do not silently substitute assumptions presented in this notebook for lecture-specific instructions.

In [ ]:
# Automated verification of the notebook's key numerical claims.
assert (-3*1 - 2*1 -1) == -6
assert np.allclose(np.array([-3.,-2.])+np.array([1.,1.]),[-2.,-1.])
assert math.isclose((1-math.exp(6))/(1+math.exp(6)), math.tanh(-3), abs_tol=1e-12)
assert np.array_equal(two_layer_predict(np.vstack([class1,class2]))[2],np.array([0,0,0,1,1]))
assert math.isclose(x3_nlms,4.0)
print('All fixed arithmetic and classification checks passed.')